# Save global PM<sub>2.5</sub> mortality in single file

Due to memory constraints, global mortality is saved on an annual basis for each mortality outcome. This script combines yearly files into one and calculates the total mortality based on all health outcomes.

In [ ]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# Number of samples
n_samples = 300

In [ ]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"
# For file names
GBD_version = "GBD23"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]
dates = f"{years.start}-{years.stop}"

MORT_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "pm25" / "global" / f"{n_samples}_samples")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "pm25")

for health_VAR in health_vars:
    print(f"Processing health variable {health_VAR}")
    for ens_num in ensemble_members:
        print(f"Processing ensemble number {ens_num:02d}")

        files = f"Global_mortality_{GBD_version}_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_*.nc"
        file_path = os.path.join(MORT_DIR, files)

        ds = xr.open_mfdataset(
            sorted(glob.glob(file_path)),
            combine="nested",
            concat_dim="year")
        ds = ds.assign_coords(year=range(years.start, years.stop + 1))

        description = (f"Global {health_VAR} mortality due to PM2.5 "
                       " - scripts by A.F. Wells (2025)")
        ds.attrs["description"] = description
        ds.attrs["GBD version"] = GBD_version
        ds.attrs["model"] = model
        ds.attrs["scenario"] = scenario
        ds.attrs["ensemble_number"] = ens_num

        out_file = f"Global_mortality_{GBD_version}_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving {out_path}")
        ds.to_netcdf(out_path)

print("All processing complete.")

## Calculate the sum of all mortality outcomes

In [ ]:
for ens_num in ensemble_members:
    # Find all files for this ensemble
    in_files = f"Global_mortality_{GBD_version}_*_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(SAVE_DIR, in_files)
    files = sorted(glob.glob(in_path))

    # Open and combine
    datasets = [xr.open_dataarray(f) for f in files]

    # Align (important in case of slight coordinate mismatches)
    aligned = xr.align(*datasets, join="exact")

    # Sum across the health variables
    summed_da = sum(aligned)

    description = ("Total global mortality due to PM2.5 "
                   "- scripts by A.F. Wells (2025)")
    summed_da.attrs["description"] = description
    summed_da.attrs["GBD version"] = GBD_version
    summed_da.attrs["ensemble_number"] = ens_num
    summed_da.attrs["scenario"] = scenario
    summed_da.attrs["model"] = model

    out_file = f"Global_mortality_{GBD_version}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving summed mortality timeseries to {out_path}")
    summed_da.to_netcdf(out_path)

print("All processing complete.")